In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampNTZType, TimestampType, DoubleType, LongType

import pandas as pd

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [ ]:
df_green = spark.read.parquet("data/pq/green/*/*")
df_green.createOrReplaceTempView("green")
spark.conf.set("spark.sql.adaptive.enabled", "false")

In [ ]:
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")
df_green_revenue.show()

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-01 00:00:00|   7|            769.73|            45|
|2020-01-01 00:00:00|  17|195.02999999999997|             9|
|2020-01-01 00:00:00|  18|               7.8|             1|
|2020-01-01 00:00:00|  22|              15.8|             1|
|2020-01-01 00:00:00|  24|              87.6|             3|
|2020-01-01 00:00:00|  25|             531.0|            26|
|2020-01-01 00:00:00|  29|              61.3|             1|
|2020-01-01 00:00:00|  32| 68.94999999999999|             2|
|2020-01-01 00:00:00|  33|317.27000000000004|            11|
|2020-01-01 00:00:00|  35|129.95999999999998|             5|
|2020-01-01 00:00:00|  36|295.34000000000003|            11|
|2020-01-01 00:00:00|  37|175.67000000000002|             6|
|2020-01-01 00:00:00|  38| 98.78999999999999|             2|
|2020-01-01 00:00:00|  4

In [14]:
df_green.write.parquet("data/report/revenue/green", mode="overwrite")

In [32]:
df_yellow = spark.read.parquet("data/pq/yellow/*/*")
df_yellow.createOrReplaceTempView("yellow")
# spark.conf.set("spark.sql.adaptive.enabled", "false")

In [33]:
df_yellow_revenue = spark.sql("""
SELECT 
    date_trunc('hour', tpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")
df_yellow_revenue.show()

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-03 19:00:00| 142| 6023.089999999995|           403|
|2020-01-26 14:00:00| 239| 6541.649999999994|           437|
|2020-01-09 01:00:00| 100|            653.56|            37|
|2020-01-31 18:00:00| 237|12689.400000000009|           810|
|2020-01-04 03:00:00| 246|2092.5400000000004|           121|
|2020-01-14 09:00:00| 239| 4882.359999999997|           298|
|2020-01-27 16:00:00| 162| 7989.979999999996|           452|
|2020-01-17 20:00:00| 170| 6884.189999999997|           407|
|2020-01-23 15:00:00| 142| 5378.829999999997|           341|
|2020-01-27 06:00:00|  24|            536.49|            23|
|2020-01-01 04:00:00| 170|            2306.2|           135|
|2020-01-05 12:00:00|  68|3495.9599999999987|           235|
|2020-01-13 17:00:00| 107|4066.6799999999976|           241|
|2020-01-21 19:00:00| 16